# MOHIM Full-Song Melody ControlNet

Paper-faithful DiT ControlNet adaptation of *Editing Music with Melody and Text*. The ACE-Step backbone is frozen, its first 12 DiT blocks are copied, top-k stereo CQT is the only melody-control input, and the copied-block residuals enter the backbone through zero-initialized linear layers. The paper's no-masking ablation is used deliberately: the repeated motif condition is never dropped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'codex/dit-controlnet'
REPO_DIR = Path('/content/MOHIM')
ACESTEP_DIR = Path('/content/ACE-Step-1.5')
ACESTEP_REPOSITORY = 'https://github.com/ace-step/ACE-Step-1.5.git'
ACESTEP_REVISION = '6d467e4b5081ccb0abf1ec1bf4fdf9051a2d34b0'

if not (REPO_DIR / '.git').is_dir():
    !git clone -b "{BRANCH}" "{REPOSITORY}" "{REPO_DIR}"
%cd {REPO_DIR}
!git fetch origin "{BRANCH}"
!git switch "{BRANCH}"
!git pull --ff-only origin "{BRANCH}"

%pip uninstall -y torchao
%pip install -q bitsandbytes librosa scipy soundfile tensorboard

def write_filtered_requirements(source, destination):
    excluded = ('torchao', 'flash-attn')
    lines = source.read_text(encoding='utf-8').splitlines()
    destination.write_text('\n'.join(
        line for line in lines if not any(name in line.lower() for name in excluded)
    ) + '\n', encoding='utf-8')

root_requirements = Path('/tmp/mohim_controlnet_requirements.txt')
write_filtered_requirements(REPO_DIR / 'requirements.txt', root_requirements)
%pip install -q -r {root_requirements}

if not (ACESTEP_DIR / '.git').is_dir():
    !git clone "{ACESTEP_REPOSITORY}" "{ACESTEP_DIR}"
%cd {ACESTEP_DIR}
!git checkout "{ACESTEP_REVISION}"
ace_requirements = Path('/tmp/acestep_controlnet_requirements.txt')
write_filtered_requirements(ACESTEP_DIR / 'requirements.txt', ace_requirements)
%pip install -q -r {ace_requirements}

for source in (str(REPO_DIR), str(ACESTEP_DIR)):
    if source not in sys.path:
        sys.path.insert(0, source)
os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO_DIR), str(ACESTEP_DIR), os.environ.get('PYTHONPATH', '')])

import bitsandbytes as bnb
print('bitsandbytes:', bnb.__version__)
print('MOHIM:', REPO_DIR)
print('ACE-Step:', ACESTEP_DIR)

In [ ]:
import json, gc, shutil
import torch

VERSION = 'v4_dit_controlnet_topk_cqt_12blocks_anchored_source_audio_lr1e4'
DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MANIFEST_PATH = DATASET_DIR / 'full_song_cover_nofsq_manifest.json'
CAPTION_CACHE_PATH = DATASET_DIR / 'ace_full_song_caption_cache.json'
TARGET_TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/full_song_tensors_repeated_motif')
LEGACY_CQT_CACHE_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_topk_cqt')
CQT_CACHE_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_topk_cqt_anchored_v2')
TEXT_CACHE_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_text_conditions')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
RUN_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_runs') / VERSION
MODEL_VARIANT = 'base'
DEVICE = 'cuda'
DTYPE = torch.bfloat16
COPY_BLOCKS = 12
BATCH_SIZE = 1
DATALOADER_WORKERS = min(2, max(1, os.cpu_count() or 1))
GRADIENT_ACCUMULATION = 8
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
VALIDATION_FRACTION = 0.2
SAVE_EVERY = 1
LOG_EVERY = 10
SEED = 42
SANITY_CHECK = False
SANITY_SAMPLE_NAME = None  # None selects the first sorted full-song tensor
SANITY_STEPS = 200
SANITY_TIMESTEP = 0.5
SANITY_LEARNING_RATE = 1e-4
SANITY_DIAGNOSTIC_EVERY = 10
SANITY_PARAMETER_SAMPLES = 256

for path in (CQT_CACHE_DIR, TEXT_CACHE_DIR, RUN_DIR):
    path.mkdir(parents=True, exist_ok=True)
assert MANIFEST_PATH.is_file(), MANIFEST_PATH
assert CAPTION_CACHE_PATH.is_file(), CAPTION_CACHE_PATH
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
caption_cache = json.loads(CAPTION_CACHE_PATH.read_text(encoding='utf-8'))['captions']
target_paths = sorted(TARGET_TENSOR_DIR.glob('*.pt'))
assert len(target_paths) == manifest['metadata']['num_samples']
manifest_by_audio = {str(Path(item['audio_path']).resolve()): item for item in manifest['samples']}
print('samples:', len(target_paths), 'copy blocks:', COPY_BLOCKS)

## 1. Paper top-k CQT cache

No new occurrence search is performed. Each short motif is repeated in both directions around its stored `motif_start_sec` anchor before stereo 128-bin top-4 CQT extraction. Existing v1 CQT tensors are converted at frame resolution instead of recomputed.

In [ ]:
import hashlib
from mohim.controlnet import (
    TOPK_CQT_SCHEMA, TOPK_CQT_UNALIGNED_SCHEMA,
    align_repeated_topk_cqt, extract_repeated_topk_cqt,
)

expected_names = sorted(path.name for path in target_paths)
fingerprint = hashlib.sha256('\n'.join(expected_names).encode()).hexdigest()
success_path = CQT_CACHE_DIR / '_SUCCESS.json'
cache_complete = False
if success_path.is_file():
    success = json.loads(success_path.read_text(encoding='utf-8'))
    cache_complete = (
        success.get('schema') == TOPK_CQT_SCHEMA
        and success.get('fingerprint') == fingerprint
        and success.get('count') == len(target_paths)
        and len(list(CQT_CACHE_DIR.glob('*.pt'))) == len(target_paths)
    )

if cache_complete:
    print(f'[OK] Reusing {len(target_paths)} anchored CQT files:', CQT_CACHE_DIR)
else:
    converted = generated = 0
    for index, tensor_path in enumerate(target_paths, 1):
        output_path = CQT_CACHE_DIR / tensor_path.name
        if output_path.is_file():
            cached = torch.load(output_path, map_location='cpu', weights_only=True)
            if cached.get('schema') == TOPK_CQT_SCHEMA:
                if index % 50 == 0:
                    print(f'[CQT CHECK {index}/{len(target_paths)}]')
                continue
        target = torch.load(tensor_path, map_location='cpu', weights_only=True)
        audio_path = str(Path(target['metadata']['audio_path']).resolve())
        sample = manifest_by_audio[audio_path]
        motif_start = float(sample['motif_start_sec'])
        motif_end = float(sample['motif_end_sec'])
        duration = target['target_latents'].shape[0] / 25.0
        legacy_path = LEGACY_CQT_CACHE_DIR / tensor_path.name
        mode = 'GENERATE'
        if legacy_path.is_file():
            legacy = torch.load(legacy_path, map_location='cpu', weights_only=True)
            if legacy.get('schema') == TOPK_CQT_UNALIGNED_SCHEMA:
                melody = align_repeated_topk_cqt(
                    legacy['melody_pitch_indices'],
                    motif_start_sec=motif_start, motif_end_sec=motif_end,
                )
                mode = 'ALIGN'
                converted += 1
            else:
                legacy = None
        else:
            legacy = None
        if legacy is None:
            melody = extract_repeated_topk_cqt(
                sample['motif_seed_audio'], duration_seconds=duration,
                motif_start_sec=motif_start, motif_end_sec=motif_end,
            )
            generated += 1
        temporary = output_path.with_suffix('.pt.tmp')
        torch.save({
            'schema': TOPK_CQT_SCHEMA,
            'melody_pitch_indices': melody,
            'motif_seed_audio': sample['motif_seed_audio'],
            'motif_start_sec': motif_start,
            'motif_end_sec': motif_end,
            'duration_seconds': duration,
        }, temporary)
        temporary.replace(output_path)
        if index % 10 == 0 or index == len(target_paths):
            print(f'[CQT {mode} {index}/{len(target_paths)}] {tuple(melody.shape)}')
    assert sorted(path.name for path in CQT_CACHE_DIR.glob('*.pt')) == expected_names
    success = {'schema': TOPK_CQT_SCHEMA, 'fingerprint': fingerprint, 'count': len(target_paths)}
    temporary = success_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(success, indent=2) + '\n', encoding='utf-8')
    temporary.replace(success_path)
    print(f'[OK] aligned={converted}, generated={generated}')
print('[OK] top-k CQT cache:', CQT_CACHE_DIR)

## 2. Text-to-music condition cache

The existing caption and target latent are reused. Cover instructions are not reused; conditions are encoded with ACE-Step's text2music instruction.

In [ ]:
from acestep.constants import SFT_GEN_PROMPT, TASK_INSTRUCTIONS
from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.model_loader import load_decoder_for_training, load_text_encoder, unload_models
import time

TEXT_SCHEMA = 'controlnet_text2music_caption_condition_v1'

def build_text2music_prompt(caption, duration):
    metas = f'- bpm: N/A\n- timesignature: N/A\n- keyscale: N/A\n- duration: {duration:.1f} seconds\n'
    return SFT_GEN_PROMPT.format(TASK_INSTRUCTIONS['text2music'], caption, metas)

def format_cache_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

if SANITY_CHECK:
    selected_target = target_paths[0] if SANITY_SAMPLE_NAME is None else TARGET_TENSOR_DIR / SANITY_SAMPLE_NAME
    assert selected_target.is_file(), selected_target
    text_cache_target_paths = [selected_target]
else:
    text_cache_target_paths = target_paths

pending = [
    tensor_path for tensor_path in text_cache_target_paths
    if not (TEXT_CACHE_DIR / tensor_path.name).is_file()
]
print('selected samples:', len(text_cache_target_paths))
print('existing text conditions:', len(text_cache_target_paths) - len(pending))
print('text conditions to encode:', len(pending))
if not pending:
    print('[OK] Reusing existing text-condition caches without loading them.')

if pending:
    cache_started_at = time.monotonic()
    print('[TEXT PASS 1] loading tokenizer and text encoder...')
    tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
    pass_started_at = time.monotonic()
    for index, tensor_path in enumerate(pending, 1):
        target = torch.load(tensor_path, map_location='cpu', weights_only=True)
        sample = manifest_by_audio[str(Path(target['metadata']['audio_path']).resolve())]
        caption = caption_cache[str(sample['track_id'])]['caption']
        duration = target['target_latents'].shape[0] / 25.0
        prompt = build_text2music_prompt(caption, duration)
        text_hs, text_mask = encode_text(text_encoder, tokenizer, prompt, DEVICE, DTYPE)
        lyric_hs, lyric_mask = encode_lyrics(text_encoder, tokenizer, sample['lyrics'], DEVICE, DTYPE)
        raw_path = (TEXT_CACHE_DIR / tensor_path.name).with_suffix('.raw.pt')
        torch.save({
            'text_hidden_states': text_hs.cpu(), 'text_attention_mask': text_mask.cpu(),
            'lyric_hidden_states': lyric_hs.cpu(), 'lyric_attention_mask': lyric_mask.cpu(),
        }, raw_path)
        pass_elapsed = time.monotonic() - pass_started_at
        seconds_per_sample = pass_elapsed / index
        pass_eta = seconds_per_sample * (len(pending) - index)
        overall_eta = pass_eta + seconds_per_sample * len(pending)
        print(
            f'[TEXT PASS 1 {index}/{len(pending)}] elapsed={format_cache_duration(pass_elapsed)}, '
            f'pass ETA={format_cache_duration(pass_eta)}, overall ETA≈{format_cache_duration(overall_eta)}'
        )
    unload_models(text_encoder)
    del text_encoder, tokenizer
    gc.collect(); torch.cuda.empty_cache()

    print('[TEXT PASS 2] loading decoder/encoder model...')
    condition_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
    pass_started_at = time.monotonic()
    for index, tensor_path in enumerate(pending, 1):
        output_path = TEXT_CACHE_DIR / tensor_path.name
        raw_path = output_path.with_suffix('.raw.pt')
        raw = torch.load(raw_path, map_location='cpu', weights_only=True)
        encoder_hs, encoder_mask = run_encoder(
            condition_model,
            raw['text_hidden_states'].to(DEVICE, DTYPE), raw['text_attention_mask'].to(DEVICE),
            raw['lyric_hidden_states'].to(DEVICE, DTYPE), raw['lyric_attention_mask'].to(DEVICE),
            DEVICE, DTYPE,
        )
        temporary = output_path.with_suffix('.pt.tmp')
        torch.save({
            'schema': TEXT_SCHEMA,
            'encoder_hidden_states': encoder_hs.squeeze(0).cpu(),
            'encoder_attention_mask': encoder_mask.squeeze(0).cpu(),
        }, temporary)
        temporary.replace(output_path)
        raw_path.unlink()
        pass_elapsed = time.monotonic() - pass_started_at
        seconds_per_sample = pass_elapsed / index
        pass_eta = seconds_per_sample * (len(pending) - index)
        total_elapsed = time.monotonic() - cache_started_at
        print(
            f'[TEXT PASS 2 {index}/{len(pending)}] elapsed={format_cache_duration(total_elapsed)}, '
            f'pass ETA={format_cache_duration(pass_eta)}, overall ETA≈{format_cache_duration(pass_eta)}'
        )
    unload_models(condition_model)
    del condition_model
    gc.collect(); torch.cuda.empty_cache()

missing_outputs = [
    tensor_path.name for tensor_path in text_cache_target_paths
    if not (TEXT_CACHE_DIR / tensor_path.name).is_file()
]
assert not missing_outputs, missing_outputs[:10]
print(f'[OK] text2music condition cache: {TEXT_CACHE_DIR} ({len(text_cache_target_paths)} selected samples)')

## 3. Load the frozen 24-block ACE DiT and clone its first 12 blocks

In [ ]:
from torch.utils.data import DataLoader
import time
from mohim.controlnet import AceStepDiTControlNet
from mohim.controlnet_training import (
    ControlNetTensorDataset, collate_controlnet_batch,
    move_batch, split_paths,
)

LOCAL_ROOT = Path('/content/mohim_controlnet_data')
LOCAL_TARGET_DIR = LOCAL_ROOT / 'targets'
LOCAL_CQT_DIR = LOCAL_ROOT / 'cqt'
LOCAL_TEXT_DIR = LOCAL_ROOT / 'text'

def copy_missing_cache_files(source_dir, local_dir, names):
    local_dir.mkdir(parents=True, exist_ok=True)
    missing = [name for name in names if not (local_dir / name).is_file()]
    if not missing:
        print(f'[REUSE] {local_dir.name}: {len(names)}/{len(names)} files')
        return
    started_at = time.monotonic()
    for index, name in enumerate(missing, 1):
        source = source_dir / name
        destination = local_dir / name
        temporary = destination.with_suffix(destination.suffix + '.tmp')
        assert source.is_file(), source
        shutil.copy2(source, temporary)
        temporary.replace(destination)
        if index % 25 == 0 or index == len(missing):
            elapsed = time.monotonic() - started_at
            eta = elapsed / index * (len(missing) - index)
            print(
                f'[COPY {local_dir.name} {index}/{len(missing)}] '
                f'elapsed={elapsed / 60:.1f}m, ETA≈{eta / 60:.1f}m'
            )

if SANITY_CHECK:
    selected_source = target_paths[0] if SANITY_SAMPLE_NAME is None else TARGET_TENSOR_DIR / SANITY_SAMPLE_NAME
    assert selected_source.is_file(), selected_source
    selected_names = [selected_source.name]
else:
    selected_names = [path.name for path in target_paths]
for source_dir, local_dir in (
    (TARGET_TENSOR_DIR, LOCAL_TARGET_DIR),
    (CQT_CACHE_DIR, LOCAL_CQT_DIR),
    (TEXT_CACHE_DIR, LOCAL_TEXT_DIR),
):
    copy_missing_cache_files(source_dir, local_dir, selected_names)
local_paths = [LOCAL_TARGET_DIR / name for name in selected_names]
if SANITY_CHECK:
    sanity_path = local_paths[0] if SANITY_SAMPLE_NAME is None else LOCAL_TARGET_DIR / SANITY_SAMPLE_NAME
    assert sanity_path.is_file(), sanity_path
    train_dataset = ControlNetTensorDataset(LOCAL_TARGET_DIR, LOCAL_CQT_DIR, LOCAL_TEXT_DIR, paths=[sanity_path])
    validation_dataset = None
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_controlnet_batch)
    validation_loader = None
else:
    train_paths, validation_paths = split_paths(local_paths, validation_fraction=VALIDATION_FRACTION, seed=SEED)
    train_dataset = ControlNetTensorDataset(LOCAL_TARGET_DIR, LOCAL_CQT_DIR, LOCAL_TEXT_DIR, paths=train_paths)
    validation_dataset = ControlNetTensorDataset(LOCAL_TARGET_DIR, LOCAL_CQT_DIR, LOCAL_TEXT_DIR, paths=validation_paths)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=DATALOADER_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
        collate_fn=collate_controlnet_batch,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=DATALOADER_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=2,
        collate_fn=collate_controlnet_batch,
    )
loaded_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
base_decoder = loaded_model.decoder
loaded_model.decoder = torch.nn.Identity()
del loaded_model
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=True).to(DEVICE, DTYPE)
controlnet.train()
print('frozen blocks:', len(controlnet.base_decoder.layers))
print('copied trainable blocks:', len(controlnet.control_blocks))
print('trainable parameters:', f'{controlnet.trainable_parameter_count()/1e6:.1f}M')
validation_count = 0 if validation_dataset is None else len(validation_dataset)
print('train/validation:', len(train_dataset), validation_count)
print('dataloader workers/pin/prefetch:', DATALOADER_WORKERS, True, 2)
if SANITY_CHECK:
    print('[SANITY] one uncropped full-song sample:', train_dataset.paths[0])

## 4. L4 memory smoke test

This performs a real AdamW8bit optimizer step so its state is included. It never reduces `COPY_BLOCKS=12` automatically.

In [ ]:
from mohim.controlnet_training import memory_smoke_test

smoke_batch = move_batch(next(iter(train_loader)), DEVICE, DTYPE)
smoke_result = memory_smoke_test(
    controlnet, smoke_batch,
    timestep_mu=controlnet.base_decoder.config.timestep_mu,
    timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
    learning_rate=LEARNING_RATE,
)
print(json.dumps(smoke_result, indent=2))
assert smoke_result.get('peak_allocated_gib', 0) < 22.0, smoke_result

# The smoke step changed the copied weights. Recreate all 12 copies from the pristine frozen backbone.
base_decoder = controlnet.base_decoder
controlnet.base_decoder = torch.nn.Identity()
del controlnet, smoke_batch
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=True).to(DEVICE, DTYPE)
controlnet.train()
print('[OK] 12-block ControlNet memory smoke test passed and weights were reset')

## 5. One-sample fixed-input sanity check

With `SANITY_CHECK=True`, this first tries to memorize one uncropped full song using fixed noise and `t=0.5`. It logs staged gradient flow, sampled bf16 parameter changes, and real-vs-shuffled motif sensitivity without cloning the model.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from mohim.controlnet_training import (
    create_adamw8bit, write_history,
)

def control_parameter_groups(model):
    groups = {'after_proj': [], 'before_proj': [], 'copied_blocks': [], 'melody_encoder': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if '.after_proj.' in name:
            groups['after_proj'].append((name, parameter))
        elif '.before_proj.' in name:
            groups['before_proj'].append((name, parameter))
        elif '.copied_block.' in name:
            groups['copied_blocks'].append((name, parameter))
        elif name.startswith('melody_encoder.'):
            groups['melody_encoder'].append((name, parameter))
        else:
            raise RuntimeError(f'Unclassified trainable parameter: {name}')
    if any(not values for values in groups.values()):
        raise RuntimeError({name: len(values) for name, values in groups.items()})
    return groups

def group_grad_norms(groups):
    result = {}
    for group_name, values in groups.items():
        squared_norm = None
        tensors_with_grad = 0
        for _, parameter in values:
            if parameter.grad is None:
                continue
            tensors_with_grad += 1
            value = parameter.grad.detach().float().square().sum()
            squared_norm = value if squared_norm is None else squared_norm + value
        norm = 0.0 if squared_norm is None else float(squared_norm.sqrt())
        result[group_name] = {'norm': norm, 'tensors_with_grad': tensors_with_grad}
    return result

def take_parameter_samples(groups, sample_count):
    snapshots = {}
    for group_name, values in groups.items():
        name, parameter = max(values, key=lambda item: item[1].numel())
        count = min(sample_count, parameter.numel())
        indices = torch.linspace(0, parameter.numel() - 1, count, device=parameter.device).long()
        sampled = parameter.detach().reshape(-1).index_select(0, indices).float().cpu()
        snapshots[group_name] = {'name': name, 'indices': indices.cpu(), 'values': sampled}
    return snapshots

def compare_parameter_samples(model, before):
    result = {}
    named_parameters = dict(model.named_parameters())
    for group_name, snapshot in before.items():
        parameter = named_parameters[snapshot['name']]
        indices = snapshot['indices'].to(parameter.device)
        after = parameter.detach().reshape(-1).index_select(0, indices).float().cpu()
        delta = (after - snapshot['values']).abs()
        result[group_name] = {
            'sample_parameter': snapshot['name'],
            'max_abs_delta': float(delta.max()),
            'changed_fraction': float((delta > 0).float().mean()),
        }
    return result

def fixed_prediction(model, batch, noise, timestep, melody):
    target = batch['target_latents']
    amount = timestep[:, None, None]
    noised = amount * noise + (1.0 - amount) * target
    context = batch['context_latents']
    return model(
        hidden_states=noised, timestep=timestep, timestep_r=timestep,
        attention_mask=batch['attention_mask'],
        encoder_hidden_states=batch['encoder_hidden_states'],
        encoder_attention_mask=batch['encoder_attention_mask'],
        context_latents=context, melody_pitch_indices=melody,
    )[0]

def fixed_flow_matching_loss(model, batch, noise, timestep):
    target = batch['target_latents']
    prediction = fixed_prediction(
        model, batch, noise, timestep, batch['melody_pitch_indices'],
    )
    flow = noise - target
    weights = batch['attention_mask'].to(prediction.dtype).unsqueeze(-1)
    return ((prediction - flow).square() * weights).sum() / (weights.sum() * target.shape[-1])

@torch.no_grad()
def motif_sensitivity_rms(model, batch, noise, timestep, shuffled_melody):
    was_training = model.training
    model.eval()
    with torch.autocast(device_type='cuda', dtype=DTYPE):
        real = fixed_prediction(model, batch, noise, timestep, batch['melody_pitch_indices'])
        shuffled = fixed_prediction(model, batch, noise, timestep, shuffled_melody)
    if was_training:
        model.train()
    return float((real.float() - shuffled.float()).square().mean().sqrt())

def run_fixed_one_sample_sanity():
    sanity_dir = RUN_DIR / 'sanity_one_sample_fixed'
    sanity_dir.mkdir(parents=True, exist_ok=True)
    history_path = sanity_dir / 'history.json'
    control_path = sanity_dir / 'controlnet.pt'
    writer = SummaryWriter(log_dir=str(sanity_dir / 'tensorboard'))
    groups = control_parameter_groups(controlnet)

    # Keep training mode/checkpointing, but remove stochastic dropout from the fixed-input test.
    for module in controlnet.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = 0.0
        if isinstance(getattr(module, 'attention_dropout', None), (int, float)):
            module.attention_dropout = 0.0

    batch = move_batch(next(iter(train_loader)), DEVICE, DTYPE)
    noise_generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    fixed_noise = torch.randn(
        batch['target_latents'].shape, generator=noise_generator, device=DEVICE, dtype=DTYPE,
    )
    fixed_timestep = torch.full(
        (batch['target_latents'].shape[0],), SANITY_TIMESTEP, device=DEVICE, dtype=DTYPE,
    )
    shuffle_generator = torch.Generator().manual_seed(SEED)
    permutation = torch.randperm(
        batch['melody_pitch_indices'].shape[1], generator=shuffle_generator,
    ).to(DEVICE)
    shuffled_melody = batch['melody_pitch_indices'].index_select(1, permutation)
    optimizer = create_adamw8bit(
        controlnet, learning_rate=SANITY_LEARNING_RATE, weight_decay=0.0,
    )
    history = []
    ever_changed = {name: False for name in groups}
    first_fixed_loss = None
    initial_sensitivity = motif_sensitivity_rms(
        controlnet, batch, fixed_noise, fixed_timestep, shuffled_melody,
    )
    sample_path = batch['paths'][0]
    full_length = batch['target_latents'].shape[1]
    print(f'[SANITY] sample={sample_path} | full length={full_length} | bf16 | fixed t={SANITY_TIMESTEP} | lr={SANITY_LEARNING_RATE:g}')
    print(f'[SANITY step 0] real-vs-shuffled output RMS={initial_sensitivity:.8g}')

    for step in range(1, SANITY_STEPS + 1):
        controlnet.train()
        optimizer.zero_grad(set_to_none=True)
        before = take_parameter_samples(groups, SANITY_PARAMETER_SAMPLES)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            loss = fixed_flow_matching_loss(
                controlnet, batch, fixed_noise, fixed_timestep,
            )
        loss.backward()
        grad_stats = group_grad_norms(groups)
        preclip_total = torch.nn.utils.clip_grad_norm_(
            list(controlnet.trainable_parameters()), MAX_GRAD_NORM,
        )
        optimizer.step()
        delta_stats = compare_parameter_samples(controlnet, before)
        for group_name in groups:
            ever_changed[group_name] |= delta_stats[group_name]['max_abs_delta'] > 0.0
        should_diagnose = (
            step <= 3 or step % SANITY_DIAGNOSTIC_EVERY == 0 or step == SANITY_STEPS
        )
        sensitivity = (
            motif_sensitivity_rms(
                controlnet, batch, fixed_noise, fixed_timestep, shuffled_melody,
            ) if should_diagnose else None
        )
        row = {
            'step': step, 'loss': float(loss.detach()),
            'preclip_total_grad_norm': float(preclip_total),
            'motif_sensitivity_rms': sensitivity,
        }
        if first_fixed_loss is None:
            first_fixed_loss = row['loss']
        for group_name in groups:
            row[f'grad_norm/{group_name}'] = grad_stats[group_name]['norm']
            row[f'grad_tensors/{group_name}'] = grad_stats[group_name]['tensors_with_grad']
            row[f'param_max_delta/{group_name}'] = delta_stats[group_name]['max_abs_delta']
            row[f'param_changed_fraction/{group_name}'] = delta_stats[group_name]['changed_fraction']
            row[f'sample_parameter/{group_name}'] = delta_stats[group_name]['sample_parameter']
            writer.add_scalar(f'grad_norm/{group_name}', grad_stats[group_name]['norm'], step)
            writer.add_scalar(f'param_max_delta/{group_name}', delta_stats[group_name]['max_abs_delta'], step)
            writer.add_scalar(f'param_changed_fraction/{group_name}', delta_stats[group_name]['changed_fraction'], step)
        writer.add_scalar('loss/fixed', row['loss'], step)
        writer.add_scalar('grad_norm/preclip_total', row['preclip_total_grad_norm'], step)
        if sensitivity is not None:
            writer.add_scalar('condition/real_vs_shuffled_output_rms', sensitivity, step)
        history.append(row)
        if should_diagnose:
            write_history(history_path, history)
            grad_text = ', '.join(
                '{}={:.3e}'.format(name, grad_stats[name]['norm']) for name in groups
            )
            delta_text = ', '.join(
                '{}={:.3e}'.format(name, delta_stats[name]['max_abs_delta']) for name in groups
            )
            print('[SANITY step {}/{}] loss={:.6f} preclip={:.3e} sensitivity={:.3e}'.format(step, SANITY_STEPS, row['loss'], row['preclip_total_grad_norm'], sensitivity))
            print('  grad norm:', grad_text)
            print('  sampled max |delta parameter|:', delta_text)
            if step == 1 and grad_stats['after_proj']['norm'] == 0.0:
                print('[CHECK] after_proj grad is zero: inspect residual injection and autograd connectivity.')
            if step == 1 and grad_stats['after_proj']['norm'] > 0.0 and not ever_changed['after_proj']:
                print('[CHECK] after_proj has grad but sampled bf16 weights did not move: inspect LR/optimizer update resolution.')
            if step == 2 and (grad_stats['copied_blocks']['norm'] == 0.0 or grad_stats['before_proj']['norm'] == 0.0):
                print('[CHECK] copied block/before_proj staged gradient is still zero after after_proj opened.')
            if step == 3 and grad_stats['melody_encoder']['norm'] == 0.0:
                print('[CHECK] melody encoder staged gradient is still zero after before_proj opened.')
            if step >= 3 and sensitivity == 0.0:
                print('[CHECK] real and shuffled motifs still produce identical outputs: condition path is ineffective.')
            if step == SANITY_DIAGNOSTIC_EVERY and ever_changed['after_proj'] and row['loss'] >= first_fixed_loss - 1e-4:
                print('[CHECK] after_proj moved but fixed loss has not decreased: inspect residual use and loss wiring.')
            writer.flush()
        del loss, before

    torch.save(controlnet.control_state_dict(), control_path)
    writer.close()
    optimizer.zero_grad(set_to_none=True)
    del optimizer, batch, fixed_noise, fixed_timestep, shuffled_melody
    gc.collect(); torch.cuda.empty_cache()
    print('[OK] sanity diagnostics:', history_path)

if SANITY_CHECK:
    run_fixed_one_sample_sanity()
else:
    print('[SKIP] Set SANITY_CHECK=True and rerun the data/model cells for the one-sample sanity check.')

## 6. Full-dataset training

This cell is intentionally separate from the sanity check. It runs only after setting `SANITY_CHECK=False` and rerunning the data/model setup cells.

In [ ]:
import hashlib
import time
from torch.utils.tensorboard import SummaryWriter
from mohim.controlnet_training import (
    create_adamw8bit, create_inverse_lr, flow_matching_step,
    load_training_checkpoint, save_training_checkpoint, write_history,
)

if SANITY_CHECK:
    print('[SKIP] Full-dataset training is disabled while SANITY_CHECK=True.')

LATEST_PATH = RUN_DIR / 'latest.pt'
EPOCH_CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
EPOCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_PROTOCOL = 'fixed_noise_timestep_v1'
BEST_CONTROL_PATH = RUN_DIR / 'best_controlnet_fixed_validation.pt'
HISTORY_PATH = RUN_DIR / 'history.json'
optimizer = None
scheduler = None
if not SANITY_CHECK:
    optimizer = create_adamw8bit(controlnet, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = create_inverse_lr(optimizer, inverse_gamma=10_000, power=0.5)
start_epoch, global_step, best_validation_loss, history = 1, 0, float('inf'), []
epoch_checkpoint_paths = sorted(EPOCH_CHECKPOINT_DIR.glob('epoch_*.pt'))
RESUME_PATH = epoch_checkpoint_paths[-1] if epoch_checkpoint_paths else LATEST_PATH
if not SANITY_CHECK and RESUME_PATH.is_file():
    state = load_training_checkpoint(RESUME_PATH, model=controlnet, optimizer=optimizer, scheduler=scheduler)
    start_epoch = int(state['epoch']) + 1
    global_step = int(state['global_step'])
    history = list(state['history'])
    print('resume checkpoint:', RESUME_PATH)
    print('resume epoch:', start_epoch)
fixed_validation_rows = [
    row for row in history
    if row.get('validation_protocol') == VALIDATION_PROTOCOL
]
best_validation_loss = min(
    (float(row['validation_loss']) for row in fixed_validation_rows),
    default=float('inf'),
)
if not SANITY_CHECK:
    if fixed_validation_rows:
        print(f'[VALIDATION] resumed fixed-protocol best: {best_validation_loss:.6f}')
    else:
        print('[VALIDATION] starting fixed noise/timestep; prior random validation is excluded from best selection')
writer = None if SANITY_CHECK else SummaryWriter(log_dir=str(RUN_DIR / 'tensorboard'))

def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def fixed_validation_inputs(batch):
    target = batch['target_latents']
    noises, timesteps = [], []
    timestep_mu = controlnet.base_decoder.config.timestep_mu
    timestep_sigma = controlnet.base_decoder.config.timestep_sigma
    for sample_index, sample_path in enumerate(batch['paths']):
        key = f'{SEED}:{Path(sample_path).name}'.encode('utf-8')
        sample_seed = int.from_bytes(hashlib.sha256(key).digest()[:8], 'little') % (2**63 - 1)
        noise_generator = torch.Generator(device=target.device).manual_seed(sample_seed)
        valid_length = int(torch.count_nonzero(batch['attention_mask'][sample_index]).item())
        sample_noise = torch.zeros_like(target[sample_index])
        sample_noise[:valid_length] = torch.randn(
            (valid_length, *target.shape[2:]), generator=noise_generator,
            device=target.device, dtype=target.dtype,
        )
        noises.append(sample_noise)
        timestep_generator = torch.Generator(device=target.device).manual_seed(
            (sample_seed + 1) % (2**63 - 1)
        )
        normal = torch.randn(
            (1,), generator=timestep_generator, device=target.device, dtype=target.dtype,
        )[0]
        timesteps.append(torch.sigmoid(normal * timestep_sigma + timestep_mu))
    return torch.stack(noises), torch.stack(timesteps)

training_started_at = time.monotonic()

for epoch in (() if SANITY_CHECK else range(start_epoch, EPOCHS + 1)):
    epoch_started_at = time.monotonic()
    controlnet.train()
    optimizer.zero_grad(set_to_none=True)
    train_sum = 0.0
    for batch_index, cpu_batch in enumerate(train_loader, 1):
        batch = move_batch(cpu_batch, DEVICE, DTYPE)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            loss = flow_matching_step(
                controlnet, batch,
                timestep_mu=controlnet.base_decoder.config.timestep_mu,
                timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
            )
        (loss / GRADIENT_ACCUMULATION).backward()
        train_sum += float(loss.detach())
        should_step = batch_index % GRADIENT_ACCUMULATION == 0 or batch_index == len(train_loader)
        if should_step:
            torch.nn.utils.clip_grad_norm_(list(controlnet.trainable_parameters()), MAX_GRAD_NORM)
            optimizer.step(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
            writer.add_scalar('loss/train_step', float(loss.detach()), global_step)
            writer.add_scalar('learning_rate', scheduler.get_last_lr()[0], global_step)
        if batch_index % LOG_EVERY == 0:
            elapsed = time.monotonic() - epoch_started_at
            seconds_per_batch = elapsed / batch_index
            epoch_eta = seconds_per_batch * (len(train_loader) - batch_index)
            remaining_batches = (
                len(train_loader) - batch_index
                + len(validation_loader)
                + (EPOCHS - epoch) * (len(train_loader) + len(validation_loader))
            )
            total_eta = seconds_per_batch * remaining_batches
            print(
                f'Epoch {epoch}/{EPOCHS}, batch {batch_index}/{len(train_loader)}, '
                f'loss={float(loss):.4f}, elapsed={format_duration(elapsed)}, '
                f'epoch ETA={format_duration(epoch_eta)}, total ETA≈{format_duration(total_eta)}'
            )
        del batch, cpu_batch, loss

    controlnet.eval()
    validation_sum = 0.0
    with torch.no_grad():
        for cpu_batch in validation_loader:
            batch = move_batch(cpu_batch, DEVICE, DTYPE)
            fixed_noise, fixed_timestep = fixed_validation_inputs(batch)
            with torch.autocast(device_type='cuda', dtype=DTYPE):
                validation_loss = flow_matching_step(
                    controlnet, batch,
                    timestep_mu=controlnet.base_decoder.config.timestep_mu,
                    timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
                    noise=fixed_noise, timestep=fixed_timestep,
                )
            validation_sum += float(validation_loss)
            del batch, cpu_batch, fixed_noise, fixed_timestep, validation_loss
    train_loss = train_sum / len(train_loader)
    validation_loss = validation_sum / len(validation_loader)
    row = {
        'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss,
        'validation_protocol': VALIDATION_PROTOCOL,
    }
    history.append(row)
    writer.add_scalar('loss/train_epoch', train_loss, epoch)
    writer.add_scalar('loss/validation', validation_loss, epoch)
    writer.flush()
    write_history(HISTORY_PATH, history)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(controlnet.control_state_dict(), BEST_CONTROL_PATH)
    if epoch % SAVE_EVERY == 0:
        epoch_checkpoint_path = EPOCH_CHECKPOINT_DIR / f'epoch_{epoch:04d}.pt'
        save_training_checkpoint(
            epoch_checkpoint_path, model=controlnet, optimizer=optimizer, scheduler=scheduler,
            epoch=epoch, global_step=global_step, best_validation_loss=best_validation_loss, history=history,
        )
        print('[CHECKPOINT]', epoch_checkpoint_path)
    epoch_seconds = time.monotonic() - epoch_started_at
    completed_epochs = epoch - start_epoch + 1
    average_epoch_seconds = (time.monotonic() - training_started_at) / completed_epochs
    total_eta = average_epoch_seconds * (EPOCHS - epoch)
    print(
        f'[OK] Epoch {epoch}/{EPOCHS}, train={train_loss:.4f}, '
        f'validation={validation_loss:.4f}, best={best_validation_loss:.4f}, '
        f'epoch time={format_duration(epoch_seconds)}, total ETA≈{format_duration(total_eta)}'
    )
if writer is not None:
    writer.close()

## 7. Revalidate saved checkpoints

Run this cell only while training is stopped. It evaluates every preserved epoch checkpoint, plus the legacy best/latest files when present, using validation-sample-specific fixed noise and timestep.

In [ ]:
import hashlib
import time
from mohim.controlnet_training import flow_matching_step

REVALIDATE_SAVED_CHECKPOINTS = False
REVALIDATION_LATEST_PATH = RUN_DIR / 'latest.pt'
REVALIDATION_HISTORY_PATH = RUN_DIR / 'history.json'
REVALIDATION_PATH = RUN_DIR / 'fixed_checkpoint_revalidation.json'

def format_revalidation_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def revalidation_inputs(batch):
    target = batch['target_latents']
    noises, timesteps = [], []
    timestep_mu = controlnet.base_decoder.config.timestep_mu
    timestep_sigma = controlnet.base_decoder.config.timestep_sigma
    for sample_index, sample_path in enumerate(batch['paths']):
        key = f'{SEED}:{Path(sample_path).name}'.encode('utf-8')
        sample_seed = int.from_bytes(hashlib.sha256(key).digest()[:8], 'little') % (2**63 - 1)
        noise_generator = torch.Generator(device=target.device).manual_seed(sample_seed)
        valid_length = int(torch.count_nonzero(batch['attention_mask'][sample_index]).item())
        sample_noise = torch.zeros_like(target[sample_index])
        sample_noise[:valid_length] = torch.randn(
            (valid_length, *target.shape[2:]), generator=noise_generator,
            device=target.device, dtype=target.dtype,
        )
        noises.append(sample_noise)
        timestep_generator = torch.Generator(device=target.device).manual_seed(
            (sample_seed + 1) % (2**63 - 1)
        )
        normal = torch.randn(
            (1,), generator=timestep_generator, device=target.device, dtype=target.dtype,
        )[0]
        timesteps.append(torch.sigmoid(normal * timestep_sigma + timestep_mu))
    return torch.stack(noises), torch.stack(timesteps)

def load_control_checkpoint_for_validation(path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if 'controlnet' in payload:
        epoch = int(payload.get('epoch', -1))
        control_state = payload['controlnet']
    else:
        epoch = -1
        control_state = payload
    controlnet.load_control_state_dict(control_state)
    del payload, control_state
    gc.collect(); torch.cuda.empty_cache()
    return epoch

@torch.no_grad()
def evaluate_fixed_validation(checkpoint_label):
    controlnet.eval()
    total = 0.0
    started_at = time.monotonic()
    for batch_index, cpu_batch in enumerate(validation_loader, 1):
        batch = move_batch(cpu_batch, DEVICE, DTYPE)
        fixed_noise, fixed_timestep = revalidation_inputs(batch)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            loss = flow_matching_step(
                controlnet, batch,
                timestep_mu=controlnet.base_decoder.config.timestep_mu,
                timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
                noise=fixed_noise, timestep=fixed_timestep,
            )
        total += float(loss)
        if batch_index % 10 == 0 or batch_index == len(validation_loader):
            elapsed = time.monotonic() - started_at
            eta = elapsed / batch_index * (len(validation_loader) - batch_index)
            print(
                f'[REVALIDATE {checkpoint_label}] {batch_index}/{len(validation_loader)}, '
                f'elapsed={format_revalidation_duration(elapsed)}, ETA≈{format_revalidation_duration(eta)}'
            )
        del batch, cpu_batch, fixed_noise, fixed_timestep, loss
    return total / len(validation_loader)

if REVALIDATE_SAVED_CHECKPOINTS:
    epoch_paths = sorted((RUN_DIR / 'checkpoints').glob('epoch_*.pt'))
    candidates = [('epoch_checkpoint', path, None) for path in epoch_paths]
    if REVALIDATION_LATEST_PATH.is_file():
        candidates.append(('legacy_latest', REVALIDATION_LATEST_PATH, None))
    legacy_best_path = RUN_DIR / 'best_controlnet.pt'
    revalidation_history = (
        json.loads(REVALIDATION_HISTORY_PATH.read_text(encoding='utf-8'))
        if REVALIDATION_HISTORY_PATH.is_file() else []
    )
    legacy_rows = [row for row in revalidation_history if not row.get('validation_protocol')]
    legacy_best_epoch = (
        int(min(legacy_rows, key=lambda row: float(row['validation_loss']))['epoch'])
        if legacy_rows else None
    )
    if legacy_best_path.is_file():
        candidates.append(('legacy_random_best', legacy_best_path, legacy_best_epoch))
    assert candidates, f'No saved checkpoints found in {RUN_DIR}'

    results = []
    for kind, checkpoint_path, epoch_hint in candidates:
        checkpoint_epoch = load_control_checkpoint_for_validation(checkpoint_path)
        epoch_value = checkpoint_epoch if checkpoint_epoch >= 0 else epoch_hint
        label = f'{kind}:epoch={epoch_value}'
        fixed_loss = evaluate_fixed_validation(label)
        result = {
            'epoch': epoch_value, 'checkpoint': str(checkpoint_path),
            'kind': kind, 'validation_loss': fixed_loss,
            'validation_protocol': 'fixed_noise_timestep_v1',
        }
        results.append(result)
        print(f'[RESULT] {label}, fixed validation={fixed_loss:.6f}')
        REVALIDATION_PATH.write_text(json.dumps(results, indent=2), encoding='utf-8')

    restore_paths = epoch_paths or (
        [REVALIDATION_LATEST_PATH] if REVALIDATION_LATEST_PATH.is_file() else []
    )
    if restore_paths:
        restored_epoch = load_control_checkpoint_for_validation(restore_paths[-1])
        print(f'[RESTORE] epoch {restored_epoch}: {restore_paths[-1]}')
    print('[OK] fixed checkpoint revalidation:', REVALIDATION_PATH)
else:
    print('[SKIP] Set REVALIDATE_SAVED_CHECKPOINTS=True to evaluate saved checkpoints.')

## 8. Custom-motif inference

The motif audio is supplied to both the ACE-Step source input and the repeated top-k CQT ControlNet input. Both conditions are shared by the CFG branches, so CFG scales text only.

In [ ]:
import soundfile as sf
from acestep.training_v2.dual_stream_preprocess import encode_motif_condition_latents
from acestep.training_v2.model_loader import load_vae

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_CAPTION = 'Paste an ACE-Step full-song caption here.'
CUSTOM_LYRICS = 'Paste the complete lyrics here.'
DURATION_SECONDS = 180.0
CUSTOM_MOTIF_START_SECONDS = 0.0
INFERENCE_STEPS = 50
CFG_SCALE = 7.0
CONTROL_SCALE = 1.0
INFERENCE_SEED = 42
OUTPUT_PATH = RUN_DIR / 'controlnet_generated.wav'
CONTROL_CHECKPOINT_PATH = (
    RUN_DIR / 'sanity_one_sample_fixed' / 'controlnet.pt' if SANITY_CHECK else BEST_CONTROL_PATH
)
assert CUSTOM_MOTIF_PATH.is_file()
assert not CUSTOM_CAPTION.startswith('Paste')
assert not CUSTOM_LYRICS.startswith('Paste')

# Free training state before loading text encoders.
globals().pop('optimizer', None)
globals().pop('scheduler', None)
base_decoder = controlnet.base_decoder
controlnet.base_decoder = torch.nn.Identity()
del controlnet, base_decoder
gc.collect(); torch.cuda.empty_cache()

tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
prompt = build_text2music_prompt(CUSTOM_CAPTION, DURATION_SECONDS)
text_hs, text_mask = encode_text(text_encoder, tokenizer, prompt, DEVICE, DTYPE)
lyric_hs, lyric_mask = encode_lyrics(text_encoder, tokenizer, CUSTOM_LYRICS, DEVICE, DTYPE)
unload_models(text_encoder); del text_encoder, tokenizer
gc.collect(); torch.cuda.empty_cache()

custom_motif_duration = sf.info(str(CUSTOM_MOTIF_PATH)).duration
target_samples = round(DURATION_SECONDS * 48000)
vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
source_audio_latents = encode_motif_condition_latents(
    str(CUSTOM_MOTIF_PATH), vae, DTYPE, target_samples=target_samples,
    motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + custom_motif_duration,
).unsqueeze(0).to(DEVICE, DTYPE)
unload_models(vae); del vae
gc.collect(); torch.cuda.empty_cache()
source_mask = torch.ones_like(source_audio_latents)
source_context = torch.cat([source_audio_latents, source_mask], dim=-1)
print('ACE-Step source context:', tuple(source_context.shape))

loaded_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
encoder_hs, encoder_mask = run_encoder(
    loaded_model, text_hs.to(DEVICE, DTYPE), text_mask.to(DEVICE),
    lyric_hs.to(DEVICE, DTYPE), lyric_mask.to(DEVICE), DEVICE, DTYPE,
)
null_condition = loaded_model.null_condition_emb.detach()
base_decoder = loaded_model.decoder
loaded_model.decoder = torch.nn.Identity()
del loaded_model, text_hs, text_mask, lyric_hs, lyric_mask
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=False).to(DEVICE, DTYPE).eval()
controlnet.load_control_state_dict(torch.load(CONTROL_CHECKPOINT_PATH, map_location='cpu', weights_only=False))
melody = extract_repeated_topk_cqt(
    CUSTOM_MOTIF_PATH, duration_seconds=DURATION_SECONDS,
    motif_start_sec=CUSTOM_MOTIF_START_SECONDS,
    motif_end_sec=CUSTOM_MOTIF_START_SECONDS + custom_motif_duration,
).unsqueeze(0).to(DEVICE)
@torch.inference_mode()
def sample_controlnet():
    latent_length = source_audio_latents.shape[1]
    generator = torch.Generator(device=DEVICE).manual_seed(INFERENCE_SEED)
    latents = torch.randn(1, latent_length, 64, generator=generator, device=DEVICE, dtype=DTYPE)
    audio_mask = torch.ones(1, latent_length, device=DEVICE, dtype=DTYPE)
    condition = encoder_hs.to(DEVICE, DTYPE)
    condition_mask = encoder_mask.to(DEVICE)
    unconditional = null_condition.to(DEVICE, DTYPE).expand_as(condition)
    times = torch.linspace(1.0, 0.0, INFERENCE_STEPS + 1, device=DEVICE, dtype=DTYPE)
    for index in range(INFERENCE_STEPS):
        timestep = times[index].expand(2)
        velocity = controlnet(
            hidden_states=torch.cat([latents, latents]),
            timestep=timestep, timestep_r=timestep,
            attention_mask=torch.cat([audio_mask, audio_mask]),
            encoder_hidden_states=torch.cat([condition, unconditional]),
            encoder_attention_mask=torch.cat([condition_mask, condition_mask]),
            context_latents=torch.cat([source_context, source_context]),
            melody_pitch_indices=torch.cat([melody, melody]),
            control_scale=CONTROL_SCALE,
        )[0]
        conditional, unconditional_velocity = velocity.chunk(2)
        guided = unconditional_velocity + CFG_SCALE * (conditional - unconditional_velocity)
        latents = latents + (times[index + 1] - times[index]) * guided
    return latents.cpu()

generated_latents = sample_controlnet()
print('[OK] generated latents:', tuple(generated_latents.shape))

In [ ]:
import math
import torch.nn.functional as F
from IPython.display import Audio, display
from acestep.training_v2.model_loader import load_vae

del controlnet, base_decoder, encoder_hs, encoder_mask, null_condition, melody
gc.collect(); torch.cuda.empty_cache()

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    stride = chunk_frames - 2 * overlap
    decoded, factor = [], None
    for index in range(math.ceil(latents.shape[-1] / stride)):
        core_start = index * stride
        core_end = min(core_start + stride, latents.shape[-1])
        window_start = max(0, core_start - overlap)
        window_end = min(latents.shape[-1], core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        factor = factor or audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * factor)
        trim_end = round((window_end - core_end) * factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
audio = decode_latents_tiled(vae, generated_latents)
target_samples = round(DURATION_SECONDS * 48_000)
audio = F.pad(audio, (0, max(0, target_samples - audio.shape[-1])))[:, :, :target_samples]
audio = audio / audio.abs().amax().clamp_min(1.0)
sf.write(OUTPUT_PATH, audio.squeeze(0).transpose(0, 1).numpy(), 48_000)
unload_models(vae); del vae
gc.collect(); torch.cuda.empty_cache()
print(OUTPUT_PATH)
display(Audio(filename=str(OUTPUT_PATH)))